vLLM's default structured-output backend: a grammar (JSON schema, regular expression or EBNF) is compiled against a model's vocabulary, and at every step a matcher produces a bit mask of the tokens that may come next. Every vocabulary below is a toy one built in the cell: no tokenizer is downloaded, no network. Examples 1-4 run on the CPU; Example 5 needs a GPU and applies a mask there with the Triton kernel vLLM uses.

## Example 1 — the installed version; a toy vocabulary, the built-in JSON grammar, and the first token mask

In [ ]:
import importlib.metadata
import xgrammar

vocabulary = ["<s>", "</s>", "{", "}", '"', "a", "b", ":", ",", " ", "1", "2", "true", "false", "null", "[", "]", "ab", '"a"', '{"a":']
tokenizer = xgrammar.TokenizerInfo(vocabulary, stop_token_ids=[1])     # a toy 20-token vocabulary; token 1 ends the text
assert tokenizer.vocab_size == 20 and tokenizer.stop_token_ids == [1]
compiler = xgrammar.GrammarCompiler(tokenizer)
matcher = xgrammar.GrammarMatcher(compiler.compile_builtin_json_grammar())
mask = xgrammar.allocate_token_bitmask(1, tokenizer.vocab_size)
matcher.fill_next_token_bitmask(mask)
allowed = [vocabulary[i] for i in range(20) if (int(mask[0, i // 32]) >> (i % 32)) & 1]
# A JSON document here starts with an object or an array: the three tokens that begin with "{" or "[".
assert allowed == ["{", "[", '{"a":'], allowed
print("xgrammar", importlib.metadata.version("xgrammar"), "-> JSON may start with", allowed)

## Example 2 — walking a matcher: tokens, strings, the end token, roll back

In [ ]:
import xgrammar

vocabulary = ["<s>", "</s>", "{", "}", '"', "a", "b", ":", ",", " ", "1", "2", "true", "false", "null", "[", "]", "ab", '"a"', '{"a":']
tokenizer = xgrammar.TokenizerInfo(vocabulary, stop_token_ids=[1])     # a toy 20-token vocabulary; token 1 ends the text
matcher = xgrammar.GrammarMatcher(xgrammar.GrammarCompiler(tokenizer).compile_builtin_json_grammar())
assert matcher.accept_token(19)                      # the token '{"a":'
assert not matcher.accept_token(3)                   # "}" right after the colon is not JSON; the matcher stays put
assert matcher.accept_string("1}") and not matcher.is_terminated()
assert matcher.accept_token(1) and matcher.is_terminated()   # the end token finishes the document
matcher.rollback(1)                                  # undo the end token
assert not matcher.is_terminated()
matcher.reset()
assert matcher.accept_string('[true, null, {"b": [1, 2]}]') and matcher.accept_token(1) and matcher.is_terminated()
print('accepted {"a":1} token by token and a nested array as a string; a misplaced "}" was refused')

## Example 3 — grammars from a JSON schema, a regular expression and EBNF

In [ ]:
import json
import xgrammar

vocabulary = ["<s>", "</s>", "{", "}", '"', "a", "b", ":", ",", " ", "1", "2", "true", "false", "null", "[", "]", "ab", '"a"', '{"a":']
tokenizer = xgrammar.TokenizerInfo(vocabulary, stop_token_ids=[1])     # a toy 20-token vocabulary; token 1 ends the text
compiler = xgrammar.GrammarCompiler(tokenizer)
def accepts(grammar, text):
    matcher = xgrammar.GrammarMatcher(compiler.compile_grammar(grammar))
    return matcher.accept_string(text) and matcher.accept_token(1) and matcher.is_terminated()
schema = json.dumps({"type": "object", "properties": {"a": {"type": "integer"}}, "required": ["a"]})
from_schema = xgrammar.Grammar.from_json_schema(schema)
assert accepts(from_schema, '{"a": 12}') and not accepts(from_schema, '{"a": "12"}') and not accepts(from_schema, '{"b": 1}')
from_regex = xgrammar.Grammar.from_regex(r"a+b")
assert accepts(from_regex, "aaab") and not accepts(from_regex, "ba") and not accepts(from_regex, "aaa")
from_ebnf = xgrammar.Grammar.from_ebnf('root ::= "ab" | "b" root')
assert accepts(from_ebnf, "bbab") and not accepts(from_ebnf, "abb")
assert "root ::=" in str(from_regex) and "root ::=" in str(from_schema)
print("schema, regex and EBNF grammars each accept their valid texts and refuse the invalid ones")

## Example 4 — applying a token mask to logits on the CPU, and a compiled grammar that survives serialisation

In [ ]:
import torch
import xgrammar

vocabulary = ["<s>", "</s>", "{", "}", '"', "a", "b", ":", ",", " ", "1", "2", "true", "false", "null", "[", "]", "ab", '"a"', '{"a":']
tokenizer = xgrammar.TokenizerInfo(vocabulary, stop_token_ids=[1])     # a toy 20-token vocabulary; token 1 ends the text
compiler = xgrammar.GrammarCompiler(tokenizer)
compiled = compiler.compile_builtin_json_grammar()
matcher = xgrammar.GrammarMatcher(compiled)
mask = xgrammar.allocate_token_bitmask(1, tokenizer.vocab_size)
matcher.fill_next_token_bitmask(mask)
logits = torch.zeros(1, tokenizer.vocab_size)
xgrammar.apply_token_bitmask_inplace(logits, mask)
kept = [i for i in range(20) if logits[0, i] == 0]
assert kept == [2, 15, 19] and torch.isinf(logits[0, [i for i in range(20) if i not in kept]]).all(), logits
restored = xgrammar.CompiledGrammar.deserialize_json(compiled.serialize_json(), tokenizer)
again = xgrammar.GrammarMatcher(restored)
restored_mask = xgrammar.allocate_token_bitmask(1, tokenizer.vocab_size)
again.fill_next_token_bitmask(restored_mask)
assert torch.equal(restored_mask, mask)
print("3 of 20 logits kept, 17 set to -inf; the compiled grammar round-trips through JSON,", compiled.memory_size_bytes, "bytes in memory")

## Example 5 — the same mask applied on the GPU by the Triton kernel

In [ ]:
import torch
import xgrammar

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
vocabulary = ["<s>", "</s>", "{", "}", '"', "a", "b", ":", ",", " ", "1", "2", "true", "false", "null", "[", "]", "ab", '"a"', '{"a":']
tokenizer = xgrammar.TokenizerInfo(vocabulary, stop_token_ids=[1])     # a toy 20-token vocabulary; token 1 ends the text
matcher = xgrammar.GrammarMatcher(xgrammar.GrammarCompiler(tokenizer).compile_builtin_json_grammar())
mask = xgrammar.allocate_token_bitmask(2, tokenizer.vocab_size)
matcher.fill_next_token_bitmask(mask, 0)                       # row 0: the start of a document
matcher.accept_token(19)
matcher.fill_next_token_bitmask(mask, 1)                       # row 1: after '{"a":'
on_cpu = torch.zeros(2, tokenizer.vocab_size)
xgrammar.apply_token_bitmask_inplace(on_cpu, mask)
on_gpu = torch.zeros(2, tokenizer.vocab_size, device="cuda")
xgrammar.apply_token_bitmask_inplace(on_gpu, mask.to("cuda"))  # backend "auto" picks the Triton kernel for CUDA tensors
assert torch.equal(on_gpu.cpu(), on_cpu), (on_gpu, on_cpu)
assert not torch.equal(on_cpu[0], on_cpu[1])
print("GPU and CPU masking agree on both rows; row 1 keeps", [vocabulary[i] for i in range(20) if on_cpu[1, i] == 0])